In [3]:
import sys
import re
import os
import pandas as pd
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objs as go

In [4]:
## Picarro Preprocessing
def load_picarro_file(filepath,keep_cols = ['CH4_dry_sync']):
    """
    Load a Picarro data file and return a DataFrame with datetime index and CH4_dry_sync column.
    """
    usecols = ["DATE", "TIME", "CH4_dry_sync"]
    df = pd.read_csv(
        filepath,
        sep='\s+',
        usecols=usecols,
        skiprows=[1],  # skip units row if present
        dtype={"DATE": str, "TIME": str, "CH4_dry_sync": float},
        engine="python"
    )
    df["datetime"] = pd.to_datetime(df["DATE"] + " " + df["TIME"], errors="coerce")
    df = df.dropna(subset=["datetime"].append(keep_cols))
    df = df.set_index('datetime')
    df = df[keep_cols]
    for col in keep_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df[keep_cols]    

def load_all_picarro_files(input_folder):
    """
    Load all Picarro data files from the input folder and concatenate them into a single DataFrame.
    """
    all_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.dat')]
    df_list = []
    for file in all_files:
        try:
            df = load_picarro_file(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    if df_list:
        return pd.concat(df_list).sort_index()
    else:
        return pd.DataFrame()

input_folder = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/Wyoming/full_picarro'
output_file = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/full_preprocessed/picarro.csv'
full_picarro_df = load_all_picarro_files(input_folder)
full_picarro_df.to_csv(output_file)

In [10]:
# Aeris 460 preprocessing
def load_ultra_file(filepath, keep_cols=['CH4 (ppm)', 'C2H6 (ppm)']):
    """
    Load an Aeris Ultra data file and return a DataFrame with datetime index and specified columns.
    C2H6 is converted from ppb to ppm.
    """

    df = pd.read_csv(filepath)
    df['datetime'] = pd.to_datetime(df['Time Stamp'], format="%m/%d/%Y %H:%M:%S.%f", errors='coerce')
    df['C2H6 (ppm)'] = df['C2H6 (ppb)'] / 1000.0  # convert ppb to ppm
    df = df.dropna(subset=['datetime'] + keep_cols)
    df = df.set_index('datetime')
    df = df[keep_cols]
    for col in keep_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df[keep_cols]

def load_all_ultra_files(input_folder):
    """
    Load all Aeris Ultra data files from the input folder and concatenate them into a single DataFrame.
    """
    all_files = [
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.endswith('.txt') and 'Eng' not in f and 'spectralite' not in f
    ]
    df_list = []
    for file in all_files:
        try:
            df = load_ultra_file(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    if df_list:
        return pd.concat(df_list).sort_index()
    else:
        return pd.DataFrame() 

input_folder = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/Wyoming/full_aeris460'
output_file = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/full_preprocessed/ultra460.csv'
full_ultra460_df = load_all_ultra_files(input_folder)
full_ultra460_df.to_csv(output_file)

In [8]:
# Preprocess rpi Pico
def load_pico_rpi_file(filepath, keep_cols=['CH4 (ppm)','C2H6 (ppm)']):

    df = pd.read_csv(filepath, skiprows = 5)
    df['datetime'] = pd.to_datetime(df['Epoch_time(s since 1/1/1970 UTC)'], unit='s', errors='coerce')
    df['C2H6 (ppm)'] = df['C2H6 (ppb)'] / 1000.0  # convert ppb to ppm
    df = df.dropna(subset=['datetime'] + keep_cols)
    df = df.set_index('datetime')
    df = df[keep_cols]
    for col in keep_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df[keep_cols]

def load_all_pico_rpi_files(input_folder):
    all_files = [
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.endswith('.dat') and 'pico' in f.lower()
    ]
    df_list = []
    for file in all_files:
        try:
            df = load_pico_rpi_file(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    if df_list:
        return pd.concat(df_list).sort_index()
    else:
        return pd.DataFrame()

input_folder = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/LANL'
output_file = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/full_preprocessed/pico_rpi.csv'
full_pico_rpi_df = load_all_pico_rpi_files(input_folder)
full_pico_rpi_df.to_csv(output_file)

In [11]:
# Preprocess rpi ultra
def load_ultra_rpi_file(filepath, keep_cols=['CH4 (ppm)','C2H6 (ppm)', 'C3H8 (ppm)']):

    df = pd.read_csv(filepath, skiprows = 5)
    df['datetime'] = pd.to_datetime(df['Epoch_time(s since 1/1/1970 UTC)'], unit='s', errors='coerce')
    df = df.dropna(subset=['datetime'] + keep_cols)
    df = df.set_index('datetime')
    df = df[keep_cols]
    for col in keep_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df[keep_cols]

def load_all_ultra_rpi_files(input_folder):
    all_files = [
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.endswith('.dat') and 'ultra' in f.lower()
    ]
    df_list = []
    for file in all_files:
        try:
            df = load_ultra_rpi_file(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
    if df_list:
        return pd.concat(df_list).sort_index()
    else:
        return pd.DataFrame()

input_folder = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/LANL'
output_file = '/uufs/chpc.utah.edu/common/home/lin-group24/agm/SLV_mobile_methane/2026/full_preprocessed/ultra321_rpi.csv'
full_ultra321_rpi_df = load_all_ultra_rpi_files(input_folder)
full_ultra321_rpi_df.to_csv(output_file)

In [23]:
# Rename columns for clarity
full_picarro_df = full_picarro_df.rename(columns={'CH4_dry_sync': 'ch4_picarro'})
full_ultra460_df = full_ultra460_df.rename(columns={'CH4 (ppm)': 'ch4_ultra460', 'C2H6 (ppm)': 'c2h6_ultra460'})
full_pico_rpi_df = full_pico_rpi_df.rename(columns={'CH4 (ppm)': 'ch4_pico', 'C2H6 (ppm)': 'c2h6_pico'})
full_ultra321_rpi_df = full_ultra321_rpi_df.rename(columns={'CH4 (ppm)': 'ch4_ultra321', 'C2H6 (ppm)': 'c2h6_ultra321', 'C3H8 (ppm)': 'c3h8_ultra321'})

# Create a dictionary of resampled DataFrames
dfs = {
    'picarro': full_picarro_df.resample('2s').mean(),
    'ultra460': full_ultra460_df.resample('2s').mean(),
    'pico_rpi': full_pico_rpi_df.resample('2s').mean(),
    'ultra321_rpi': full_ultra321_rpi_df.resample('2s').mean()
}

def pare_down_dfs(dfs, columns_dict=None, start=None, end=None):
    """
    Pare down each DataFrame in dfs (dict) to specified columns and datetime range.
    columns_dict: dict mapping df key to list of columns to keep, e.g. {'picarro': ['ch4_picarro']}
    start, end: datetime strings or pd.Timestamp for slicing
    Returns a dict of pared-down DataFrames.
    """
    pared = {}
    for key, df in dfs.items():
        df_p = df.copy()
        if columns_dict and key in columns_dict:
            df_p = df_p[columns_dict[key]]
        if start or end:
            df_p = df_p.loc[start:end]
        pared[key] = df_p
    return pared


In [39]:
columns_dict = {'picarro': ['ch4_picarro'], 'ultra460': ['ch4_ultra460'], 'pico_rpi': ['ch4_pico'], 'ultra321_rpi': ['ch4_ultra321']}
start = '2026-02-03 16:00:00'
end = '2026-02-13 00:00:00'
pared_dfs = pare_down_dfs(dfs, columns_dict=columns_dict, start=start, end=end)

In [40]:
pared_dfs

{'picarro':                      ch4_picarro
 datetime                        
 2026-02-03 16:59:10     2.084717
 2026-02-03 16:59:12     2.084497
 2026-02-03 16:59:14     2.084174
 2026-02-03 16:59:16     2.084065
 2026-02-03 16:59:18     2.083303
 ...                          ...
 2026-02-12 23:21:14     2.058655
 2026-02-12 23:21:16     2.069045
 2026-02-12 23:21:18     2.048302
 2026-02-12 23:21:20     2.037352
 2026-02-12 23:21:22     2.023711
 
 [400267 rows x 1 columns],
 'ultra460':                      ch4_ultra460
 datetime                         
 2026-02-03 17:26:38      2.265240
 2026-02-03 17:26:40      2.257755
 2026-02-03 17:26:42      2.283630
 2026-02-03 17:26:44      2.270100
 2026-02-03 17:26:46      2.255945
 ...                           ...
 2026-02-12 23:15:10      2.272975
 2026-02-12 23:15:12      2.265500
 2026-02-12 23:15:14      2.255975
 2026-02-12 23:15:16      2.252720
 2026-02-12 23:15:18      2.252135
 
 [399261 rows x 1 columns],
 'pico_rpi':        

In [ ]:
fig = go.Figure()
for key, df in pared_dfs.items():
    for col in df.columns:
        fig.add_trace(go.Scatter(
            x=df.index, y=df[col],
            mode='lines', name=f"{key}: {col}"
        ))
fig.update_layout(
    xaxis_title="Datetime",
    yaxis_title="Value",
    legend_title="Instrument: Variable",
    template="plotly_white"
)
fig.show()
